## Column Generation example

In [36]:
using Pkg

Pkg.activate(".")
Pkg.instantiate()

  Activating project at `c:\Users\Patricia\Downloads\iniciacao cientifica\ESPPRC_route_generation`


In [37]:
using HiGHS
using JuMP
using Graphs
using GraphPlot
using Plots
using LinearAlgebra
using Random

In [38]:
include("pdp.jl")

rota_da_solucao (generic function with 1 method)

## Problema aleatorios 

In [39]:
function problema_com_cidades_do_Parana(m, r, num_caminhoes, L)
    
    Curitiba = 1
    Maringa = 2
    Londrina = 3
    Pronta_Grossa = 4
    Cascavel = 5
    Foz_do_Iguacu = 6 
    Sao_Jose_dos_Pinhais = 7
    Paranagua = 8
    Guarapuava = 9
    Antonina = 10
    Toledo = 11
    Umuarama = 12 
    # calculando a distancia entre todas as cidades 
    C = Array{Float64}(undef, m, m)
    for i = 1:m
        C[i, i] = 0
    end
    C[1, 2] = 367
    C[1, 3] = 331
    C[1, 4] = 117
    C[1, 5] = 430
    C[1, 6] = 540
    C[1, 7] = 34
    C[1, 8] = 91
    C[1, 9] = 228
    C[1, 10] = 92
    C[1, 11] = 455
    C[1, 12] = 467

    C[2, 3] = 98
    C[2, 4] = 269
    C[2, 5] = 228
    C[2, 6] = 329
    C[2, 7] = 379
    C[2, 8] = 435
    C[2, 9] = 249
    C[2, 10] = 436
    C[2, 11] = 242
    C[2, 12] = 147

    C[3, 4] = 234
    C[3, 5] = 311
    C[3, 6] = 413 
    C[3, 7] = 348
    C[3, 8] = 402
    C[3, 9] = 275
    C[3, 10] = 406
    C[3, 11] = 323
    C[3, 12] = 232

    C[4, 5] = 354
    C[4, 6] = 461
    C[4, 7] = 136
    C[4, 8] = 199
    C[4, 9] = 155
    C[4, 10] = 201
    C[4, 11] = 378
    C[4, 12] = 373

    C[5, 6] = 124
    C[5, 7] = 447
    C[5, 8] = 511
    C[5, 9] = 220
    C[5, 10] = 512
    C[5, 11] = 40
    C[5, 12] = 138

    C[6, 7] = 549
    C[6, 8] = 615
    C[6, 9] = 320
    C[6, 10] = 616
    C[6, 11] = 138
    C[6, 12] = 236

    C[7, 8] = 84
    C[7, 9] = 234
    C[7, 10] = 86
    C[7, 11] = 472
    C[7, 12] = 490

    C[8, 9] = 295
    C[8, 10] = 58
    C[8, 11] = 530
    C[8, 12] = 553

    C[9, 10] = 314
    C[9, 11] = 238
    C[9, 12] = 267

    C[10, 11] = 537
    C[10, 12] = 575

    C[11, 12] = 111

    for t = 1:m-1
        for s = t+1:m
            
            C[s,t]=C[t, s]
        end
    end
    

    # gerando uma quantidade r de tarefas 
    task = Array{Int64}(undef, r, 2)
    for i = 1:r
        a = rand(1:m)
        b = rand(1:m)
        while b == a 
            b = rand(1:m)
        end
        task[i, 1] = a
        task[i, 2] = b
    end

    # gerando uma quantidade r de janelas de tempo para realizacao de cada tarefa 
    W = Array{Float64}(undef, r, 2)
    for i = 1:r
        origem = task[i, 1]
        destino = task[i, 2]
        distancia = C[origem, destino]  # distância entre as cidades da tarefa
        c = distancia + (L - distancia) * rand()
        d = c + (L - c) * rand()     
        W[i, 1] = c
        W[i, 2] = d
    end
    return C, task, W
end

problema_com_cidades_do_Parana (generic function with 1 method)

## Grafo

In [40]:
function grafo(C, task, r)
    # Identificar apenas as cidades usadas nas tarefas
    usados = unique(vcat(task[:, 1], task[:, 2]))
    mapa_cidade = Dict(cidade => i for (i, cidade) in enumerate(usados))

    # Criar grafo apenas com as cidades usadas
    g = SimpleDiGraph(length(usados))

    # Criar matriz truncada inicializada com zeros
    C_truncada = zeros(Float64, length(usados), length(usados))

    # Preencher matriz apenas com distâncias das tarefas
    for i = 1:r
        origem_real = task[i, 1]
        destino_real = task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        C_truncada[origem, destino] = trunc(C[origem_real, destino_real])
    end

    # Criar lista de arestas com pesos
    weights = Dict()
    for i in 1:size(task, 1)
        origem_real, destino_real = task[i, 1], task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        if C_truncada[origem, destino] > 0
            add_edge!(g, origem, destino)
            weights[(origem, destino)] = C_truncada[origem, destino]
        end
    end

    # Obter rótulos das arestas
    graph_edges = collect(Graphs.edges(g))
    edge_labels = [weights[(u.src, u.dst)] for u in graph_edges]
    
    # Plotar o grafo (números originais das cidades no rótulo)
    #gplot(g, nodelabel=usados, edgelabel=edge_labels)
    display(gplot(g, nodelabel=usados, edgelabel=edge_labels))
    return g, weights
end

grafo (generic function with 1 method)

## Dados de entrada

In [41]:
# m vai ser a quantidade de cidades dentro do Parana 12
m = 12

# r vai ser a quantidade de tarefas que vou ter 20 - 15
r = 20

# numero de caminhoes 5 - 4
num_caminhoes = 5

# Limite de tempo 7200 - 7200
L = 7200

# Semente 87213 2  
numero_primo = 	87213
teste = 2
semente = numero_primo + teste
Random.seed!(semente)

TaskLocalRNG()

In [42]:
C, task, W = problema_com_cidades_do_Parana(m, r, num_caminhoes, L)

([0.0 367.0 … 455.0 467.0; 367.0 0.0 … 242.0 147.0; … ; 455.0 242.0 … 0.0 111.0; 467.0 147.0 … 111.0 0.0], [10 5; 6 10; … ; 12 9; 6 5], [3102.0031154346716 4544.006877382281; 5814.4405619983445 6690.063219318814; … ; 5235.1919746355 6163.645696237493; 3932.8090884257854 4134.3357258739525])

In [43]:
display(C)

12×12 Matrix{Float64}:
   0.0  367.0  331.0  117.0  430.0  …   91.0  228.0   92.0  455.0  467.0
 367.0    0.0   98.0  269.0  228.0     435.0  249.0  436.0  242.0  147.0
 331.0   98.0    0.0  234.0  311.0     402.0  275.0  406.0  323.0  232.0
 117.0  269.0  234.0    0.0  354.0     199.0  155.0  201.0  378.0  373.0
 430.0  228.0  311.0  354.0    0.0     511.0  220.0  512.0   40.0  138.0
 540.0  329.0  413.0  461.0  124.0  …  615.0  320.0  616.0  138.0  236.0
  34.0  379.0  348.0  136.0  447.0      84.0  234.0   86.0  472.0  490.0
  91.0  435.0  402.0  199.0  511.0       0.0  295.0   58.0  530.0  553.0
 228.0  249.0  275.0  155.0  220.0     295.0    0.0  314.0  238.0  267.0
  92.0  436.0  406.0  201.0  512.0      58.0  314.0    0.0  537.0  575.0
 455.0  242.0  323.0  378.0   40.0  …  530.0  238.0  537.0    0.0  111.0
 467.0  147.0  232.0  373.0  138.0     553.0  267.0  575.0  111.0    0.0

In [44]:
display(task)

20×2 Matrix{Int64}:
 10   5
  6  10
  2  10
 11   6
  3   5
  8  11
 12  11
 10  11
  7   8
  5   4
  6  10
  6   9
  1  12
  8   4
  5   7
 10   5
 10  12
  9   3
 12   9
  6   5

In [45]:
display(W)

20×2 Matrix{Float64}:
 3102.0    4544.01
 5814.44   6690.06
 5709.13   6822.34
 2638.46   4801.57
 1956.63   5296.65
 4508.3    5306.06
 1073.24   7009.47
 3747.75   4035.79
  837.579  1204.7
 4936.71   6996.51
 5830.52   6443.67
 6567.27   7191.42
 7095.99   7108.09
 7182.33   7198.29
 3088.39   4624.13
 6555.68   6665.51
 2711.8    6513.8
 5598.29   6650.96
 5235.19   6163.65
 3932.81   4134.34

In [46]:
A, solucao, g = A_final(C, task, W, num_caminhoes, r)

matriz task
matriz Wu
matriz W

Iteração 1

Solução ótima=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Valor ótimo=-5.0
Iterações=5
Base=[26, 22, 23, 24, 25, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Zj - Cj (Custo reduzido final) = [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Custo arestas:
	λ0 = -1.0
	λ = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
ESPPRC: L=7198.290671096128
	S = [0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 0.0; 0.0 -1.0 0.0 -1.0 

LoadError: SingularException(16)

In [ ]:
println("É inteira? $(all(isinteger.(solucao)))")

É inteira? true


In [ ]:
rota_da_solucao(solucao, A, g)

13×3 Matrix{Float64}:
 0.0  0.0  0.0
 1.0  0.0  0.0
 0.0  0.0  0.0
 0.0  1.0  0.0
 0.0  0.0  1.0
 0.0  0.0  0.0
 1.0  0.0  0.0
 1.0  0.0  0.0
 0.0  1.0  0.0
 0.0  1.0  0.0
 1.0  0.0  0.0
 1.0  0.0  0.0
 0.0  0.0  1.0